# Publication-Ready Thyroid Cancer Analyses (Phase 3)

This notebook provides manuscript-oriented outputs:

1. **Table 1** Demographics with p-values
2. **Table 2** Tumor characteristics by stage/histology
3. **Table 3** Multivariable logistic regression (LN metastasis predictors)
4. **Figure 1** Kaplan-Meier-style Tg event curve
5. **Figure 2** FNA ROC curve proxy

Models are demonstrated with `statsmodels` and include p-values, ORs, and 95% CIs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')

DB_PATH = Path('/Users/ros/Desktop/FInal Cleaned Thyroid Data/thyroid_master.duckdb')
con = duckdb.connect(str(DB_PATH), read_only=True)
print('Connected:', DB_PATH)

## Table 1: Demographics (mean/SD, n(%), p-values)

In [ ]:
t1_base = con.execute('''
SELECT
    CASE
        WHEN disease_group = 'malignant' THEN 'Malignant'
        WHEN disease_group = 'benign' THEN 'Benign'
        ELSE 'Other'
    END AS group_name,
    TRY_CAST(age_at_surgery AS DOUBLE) AS age,
    LOWER(COALESCE(sex, 'unknown')) AS sex
FROM benign_vs_malignant_comparison
WHERE disease_group IN ('malignant', 'benign')
''').fetchdf()

summary = (
    t1_base.groupby('group_name')
    .agg(
        n=('group_name', 'count'),
        age_mean=('age', 'mean'),
        age_sd=('age', 'std'),
        female_n=('sex', lambda s: (s == 'female').sum()),
        male_n=('sex', lambda s: (s == 'male').sum())
    )
)
summary['female_pct'] = 100 * summary['female_n'] / summary['n']
summary['male_pct'] = 100 * summary['male_n'] / summary['n']

mal = t1_base[t1_base['group_name'] == 'Malignant']
ben = t1_base[t1_base['group_name'] == 'Benign']

age_p = stats.ttest_ind(
    mal['age'].dropna(),
    ben['age'].dropna(),
    equal_var=False,
    nan_policy='omit'
).pvalue

sex_ct = pd.crosstab(t1_base['group_name'], t1_base['sex'])
sex_p = stats.chi2_contingency(sex_ct).pvalue

table1 = summary.reset_index()
table1['age_mean_sd'] = table1.apply(lambda r: f"{r['age_mean']:.1f} ({r['age_sd']:.1f})", axis=1)
table1['female_n_pct'] = table1.apply(lambda r: f"{int(r['female_n'])} ({r['female_pct']:.1f}%)", axis=1)
table1['male_n_pct'] = table1.apply(lambda r: f"{int(r['male_n'])} ({r['male_pct']:.1f}%)", axis=1)
table1 = table1[['group_name', 'n', 'age_mean_sd', 'female_n_pct', 'male_n_pct']]

print(f"Age p-value (t-test): {age_p:.4g}")
print(f"Sex p-value (chi-square): {sex_p:.4g}")
table1

## Table 2: Tumor Characteristics by Stage/Histology

In [ ]:
table2 = con.execute('''
SELECT
    histology_1_type,
    overall_stage_ajcc8,
    COUNT(*) AS n,
    ROUND(AVG(largest_tumor_cm), 2) AS mean_tumor_cm,
    ROUND(STDDEV_POP(largest_tumor_cm), 2) AS sd_tumor_cm,
    ROUND(AVG(CASE WHEN ln_positive > 0 THEN 1.0 ELSE 0.0 END), 3) AS ln_positive_rate
FROM ptc_cohort
GROUP BY histology_1_type, overall_stage_ajcc8
ORDER BY n DESC
''').fetchdf()

table2.head(25)

## Table 3: Multivariable Logistic Regression (Predictors of LN Metastasis)

In [ ]:
reg_df = con.execute('''
SELECT
    TRY_CAST(age_at_surgery AS DOUBLE) AS age_at_surgery,
    LOWER(COALESCE(sex, 'unknown')) AS sex,
    TRY_CAST(largest_tumor_cm AS DOUBLE) AS largest_tumor_cm,
    COALESCE(max_tirads, 0) AS max_tirads,
    CASE
        WHEN COALESCE(ln_positive_total, 0) > 0 THEN 1
        ELSE 0
    END AS ln_met_positive
    
FROM imaging_pathology_correlation ipc
LEFT JOIN lymph_node_metastasis_view lnm
    ON ipc.research_id = lnm.research_id
WHERE TRY_CAST(largest_tumor_cm AS DOUBLE) IS NOT NULL
  AND TRY_CAST(age_at_surgery AS DOUBLE) IS NOT NULL
''').fetchdf()

reg_df = reg_df[reg_df['sex'].isin(['female', 'male'])].copy()
reg_df['sex_male'] = (reg_df['sex'] == 'male').astype(int)

model = smf.logit(
    formula='ln_met_positive ~ age_at_surgery + sex_male + largest_tumor_cm + max_tirads',
    data=reg_df
).fit(disp=False)

coef = model.params
conf = model.conf_int()
pvals = model.pvalues

or_table = pd.DataFrame({
    'term': coef.index,
    'coef': coef.values,
    'OR': np.exp(coef.values),
    'CI_2.5%': np.exp(conf[0].values),
    'CI_97.5%': np.exp(conf[1].values),
    'p_value': pvals.values
})

print(model.summary())
or_table

## Figure 1: Kaplan-Meier-Style Tg Event Curve

In [ ]:
rr = con.execute('''
SELECT
    research_id,
    tg_first_date,
    tg_last_date,
    tg_max,
    recurrence_risk_band
FROM recurrence_risk_cohort
WHERE tg_first_date IS NOT NULL
''').fetchdf()

threshold = 2.0
rr['event'] = (rr['tg_max'].fillna(0) >= threshold).astype(int)
rr['duration_days'] = (
    pd.to_datetime(rr['tg_last_date']) - pd.to_datetime(rr['tg_first_date'])
).dt.days.clip(lower=1)

def km_curve(df):
    d = df[['duration_days', 'event']].dropna().sort_values('duration_days')
    times = np.sort(d['duration_days'].unique())
    n_at_risk = len(d)
    s = 1.0
    xs = [0]
    ys = [1.0]
    for t in times:
        events = ((d['duration_days'] == t) & (d['event'] == 1)).sum()
        cens = ((d['duration_days'] == t) & (d['event'] == 0)).sum()
        if n_at_risk > 0:
            s *= (1 - events / n_at_risk)
        xs.extend([t, t])
        ys.extend([ys[-1], s])
        n_at_risk -= (events + cens)
    return np.array(xs), np.array(ys)

plt.figure(figsize=(10, 5))
for band in ['low', 'intermediate', 'high']:
    sub = rr[rr['recurrence_risk_band'] == band]
    if len(sub) == 0:
        continue
    x, y = km_curve(sub)
    plt.step(x, y, where='post', label=f'{band} (n={len(sub)})')

plt.title('Kaplan-Meier-Style Tg Event-Free Curve')
plt.xlabel('Follow-up days from first Tg')
plt.ylabel('Event-free probability')
plt.ylim(0, 1.02)
plt.legend()
plt.tight_layout()
plt.show()

## Figure 2: FNA ROC Curve Proxy

In [ ]:
roc_df = con.execute('''
WITH dx AS (
    SELECT
        research_id,
        CASE WHEN has_tumor_pathology THEN 1
             WHEN has_benign_pathology THEN 0
             ELSE NULL END AS final_malignant
    FROM master_cohort
), fna AS (
    SELECT
        research_id,
        TRY_CAST(bethesda_2023_num AS DOUBLE) AS score
    FROM fna_cytology
)
SELECT f.score, d.final_malignant
FROM fna f
JOIN dx d USING (research_id)
WHERE f.score IS NOT NULL
  AND d.final_malignant IS NOT NULL
''').fetchdf()

y_true = roc_df['final_malignant'].astype(int).values
score = roc_df['score'].values
thresholds = np.sort(np.unique(score))[::-1]

pts = []
for th in thresholds:
    y_pred = (score >= th).astype(int)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    tpr = tp / (tp + fn) if (tp + fn) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0
    pts.append((fpr, tpr, th))

roc_pts = pd.DataFrame(pts, columns=['fpr', 'tpr', 'threshold']).sort_values('fpr')
auc = np.trapz(roc_pts['tpr'], roc_pts['fpr'])

plt.figure(figsize=(6, 6))
plt.plot(roc_pts['fpr'], roc_pts['tpr'], marker='o', label=f'ROC proxy (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('FNA ROC Proxy using Bethesda 2023 Score')
plt.legend()
plt.tight_layout()
plt.show()

roc_pts.head(10)